In [13]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

import numpy as np
import pandas as pd
from soma import aims
import os

In [14]:
subject_id_list = ["sub-3823849", "sub-2521602", "sub-2455992", "sub-5911149", "sub-1889049", "sub-1834920", "sub-1086361", "sub-3389051", "sub-2579202"]
dataset_name_list = ['UkBioBank40' for i in range(9)] 
side = "L"
region = "CINGULATE." #"ORBITAL" #"CINGULATE"

In [15]:
bucket_files = []

for subject_id, dataset in zip(subject_id_list,dataset_name_list):
    if dataset.lower() in ['ukb', 'ukbiobank']:
        path = f'/neurospin/dico/data/deep_folding/current/datasets/UkBioBank/crops/2mm/{region}/mask/{side}crops'
    if dataset.lower() in ['ukb40', 'ukbiobank40']:
        path = f'/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/crops/2mm/{region}/mask/{side}crops'
    filename = f'{path}/{subject_id}_cropped_skeleton.nii.gz'#.minf'

    if os.path. isfile(filename):
        bucket_files.append(filename)
    else:
        print(f"{filename} is not a correct path, or the .bck doesn't exist")
bucket_files

['/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/crops/2mm/CINGULATE./mask/Lcrops/sub-3823849_cropped_skeleton.nii.gz',
 '/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/crops/2mm/CINGULATE./mask/Lcrops/sub-2521602_cropped_skeleton.nii.gz',
 '/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/crops/2mm/CINGULATE./mask/Lcrops/sub-2455992_cropped_skeleton.nii.gz',
 '/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/crops/2mm/CINGULATE./mask/Lcrops/sub-5911149_cropped_skeleton.nii.gz',
 '/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/crops/2mm/CINGULATE./mask/Lcrops/sub-1889049_cropped_skeleton.nii.gz',
 '/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/crops/2mm/CINGULATE./mask/Lcrops/sub-1834920_cropped_skeleton.nii.gz',
 '/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/crops/2mm/CINGULATE./mask/Lcrops/sub-1086361_cropped_skeleton.nii.gz',
 '/neurospin/dico/data/deep_folding/current/data

In [16]:
def to_bucket(obj):
    """Converts an object to a bucket if it isn't one already."""
    if obj.type() == obj.BUCKET:
        return obj
    avol = a.toAimsObject(obj)
    c = aims.Converter(intype=avol, outtype=aims.BucketMap_VOID)
    abck = c(avol)
    bck = a.toAObject(abck)
    bck.releaseAppRef()
    return bck

In [17]:
def build_gradient(pal):
    gw = ana.cpp.GradientWidget(None, 'gradientwidget', pal.header()['palette_gradients'])
    gw.setHasAlpha(True)
    nc = pal.shape[0]
    rgbp = gw.fillGradient(nc, True)
    rgb = rgbp.data()
    npal = pal.np['v']
    pb = np.frombuffer(rgb, dtype=np.uint8).reshape((nc, 4))
    npal[:, 0, 0, 0, :] = pb
    npal[:, 0, 0, 0, :3] = npal[:, 0, 0, 0, :3][:, ::-1]  # BGRA -> RGBA
    pal.update()

In [18]:
def buckets_average(subject_id_list, dataset_name_list, region, side):
    """Computes the average bucket volumes for a list of subjects."""
    dic_vol = {}
    dim = 0
    rep = 0

    if len(subject_id_list) == 0:
        return False

    # Find a valid volume for dimension checking
    while dim == 0 and rep < len(subject_id_list):
        dataset = 'UkBioBank' if dataset_name_list[rep].lower(
        ) in ['ukb', 'ukbiobank', 'projected_ukb'] else 'UkBioBank40'
        mm_skeleton_path = f"/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}crops"

        file_path = f"{mm_skeleton_path}/{subject_id_list[rep]}_cropped_skeleton.nii.gz"
        if os.path.isfile(file_path):
            sum_vol = aims.read(file_path).astype(float)
            dim = sum_vol.shape
            sum_vol.fill(0)
        else:
            print(f'FileNotFound: {file_path}')
        rep += 1

    # Process each subject
    for subject_id, dataset_name in zip(subject_id_list, dataset_name_list):
        dataset = 'UkBioBank' if dataset_name.lower(
        ) in ['ukb', 'ukbiobank', 'projected_ukb'] else 'UkBioBank40'
        mm_skeleton_path = f"/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}crops"

        file_path = f"{mm_skeleton_path}/{subject_id}_cropped_skeleton.nii.gz"
        if os.path.isfile(file_path):
            vol = aims.read(file_path)
            if vol.np.shape != dim:
                raise ValueError(
                    f"{subject_id_list[0]} and {subject_id} "
                    "must have the same dimensions")

            # Convert to binary structure
            struc3D = (vol.np > 0).astype(int)
            dic_vol[subject_id] = struc3D
            # Accumulate binary volumes
            sum_vol.np[:] += struc3D
        else:
            print(f'FileNotFound: {file_path}')

    # Normalize the accumulated volume
    sum_vol.np[:] /= len(subject_id_list)
    print(f"{sum_vol.shape}: max = {sum_vol.np.max()}")
    return sum_vol

In [20]:
sum_vol = buckets_average(subject_id_list, dataset_name_list, region, side)

block = a.createWindowsBlock(5) # nb of columns

a_sum_vol = a.toAObject(sum_vol)
a_sum_vol.setPalette(minVal=0, absoluteMode=True)
wsum = a.createWindow('Sagittal', block=block)
wsum.addObjects(a_sum_vol)
rvol = a.fusionObjects(objects=[a_sum_vol], method='VolumeRenderingFusionMethod')
rvol.releaseAppRef()
# custom palette
n = len(subject_id_list)
pal = a.createPalette('VR-palette')
pal.header()['palette_gradients'] = f'0;0.244444;0.5;1;1;1#0;0;0.535897;0.222222;1;1#0;0.7;1;0#0;0;{0.5/n};0;1;1'
build_gradient(pal)
rvol.setPalette('VR-palette', minVal=0, absoluteMode=True)
pal2 = a.createPalette('slice-palette')
pal2.header()['palette_gradients'] = f'0;0.244444;0.5;1;1;1#0;0;0.535897;0.222222;1;1#0;0.7;1;0#0;0;{0.3/n};0;{0.7/n};1;1;1'
build_gradient(pal2)
a_sum_vol.setPalette('slice-palette')
# rvol.palette().fill()
wvr = a.createWindow('3D', block=block)
wvr.addObjects(rvol)

(18, 41, 38, 1): max = 0.7777777777777778
